# Chapter 8: RAG Generation — The Answer Layer
## Topic 2: Citation and Source Attribution

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- A RAG answer's credibility depends on being able to verify where a claim actually came from. A statement like "premature withdrawal incurs a 1% penalty" is a materially different kind of statement when it's traceable to a specific policy document versus when it's an unattributed claim the model could have simply made up.
- Citation exists to close the gap between "the model said something plausible" and "the model said something that can be verified against a specific source." This is the mechanism that makes hallucination detection tractable later: if every claim is supposed to trace back to a cited source, it becomes possible to programmatically check whether it actually does.
- For a regulated domain like financial services, citation isn't just a nice-to-have user-experience feature — it's close to a compliance requirement. A customer disputing a calculated penalty needs the underlying policy document identified, not just a paraphrase from a language model.


### 2. Internal Working — Step by Step

1. **Source tagging at construction time:** each chunk in the context block carries an explicit, parseable marker identifying where it came from — something the model can reference directly in its answer.
2. **Prompting the model to cite:** the system prompt explicitly instructs the model to attribute every factual claim to its source marker whenever it states something drawn from the provided context.
3. **Two citation strategies, differing in when attribution happens:**
   - **Inline citation:** the model outputs citations as part of its natural-language answer directly. This is the simplest to implement, but citation accuracy depends entirely on the model correctly associating each claim with its source during free-form generation.
   - **Structured citation via tool use:** the model's response is a structured object with a separate field explicitly listing which source markers were used. This is more reliable to parse programmatically, especially if the surrounding system already has an established pattern for structured, tool-based model outputs.
4. **Post-hoc verification — the check that makes citation trustworthy, not just decorative:** after generation, verify that every cited source marker actually appears in the context block that was actually sent to the model. A citation to a source that was never in context is a definitive signal of model error, independent of whether the underlying factual claim happens to coincidentally be true.
5. **Surfacing citations to the end consumer:** whether a citation is shown directly to a customer, logged for a human reviewer, or used purely internally for downstream hallucination checks depends on the deployment context — customer-facing surfacing is often optional, but internal logging for every generated answer is close to mandatory in a regulated domain.


### 3. How This Is Implemented in This Project

- Building on an existing tool-use pattern already established elsewhere in the project: a new terminal tool mirrors the design of an existing "finalize" style tool — a structured way for the model to declare it's done, but now carrying both an answer and a list of cited sources as required fields in its schema.
- The schema requires two fields: the final answer text, and an explicit list of every source marker that was actually used to construct that answer. Making both fields required means the model cannot finalize an answer without also declaring what it relied on.
- The system prompt is written to instruct the model to answer using only the provided context, and to call this finalize tool with both the answer and the exact list of source markers it actually relied on.
- After the model responds, a verification step checks that every source marker in the model's declared list actually appears in the set of sources that were genuinely included in the context sent for that request — this is a simple, cheap set-membership check, not a semantic check of whether the claim is actually correct (that deeper check belongs to a later, more expensive step).


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Trusting citations without post-hoc verification is a real risk:** a model can cite a plausible-sounding but incorrect or even nonexistent source, and without an explicit verification step, this goes completely undetected.
- **Using citation strings from model output directly in file-system or database operations without validation creates an injection risk** — any value coming from model output should be treated as untrusted input until validated, the same as any other user-controllable data.
- **Citation validity is not the same thing as claim correctness:** a citation can be valid (the source genuinely was in context) while the claim attributed to it is still wrong, incomplete, or unsupported by what that source actually says. Citation verification is necessary but not sufficient — it's a cheap, syntactic check, distinct from a deeper semantic grounding check.
- **Designing a citation format the model can't reliably reproduce is a real trap** — for example, requiring exact page numbers when chunks don't actually carry that metadata sets the model up to either fail or fabricate that detail. The citation schema should only ever ask for what the pipeline actually attaches to each chunk.
- **Latency and cost:** citation verification itself is a cheap, fast, synchronous check — negligible added latency or cost compared to the generation step itself. This makes it practical to run on every single response, unlike more expensive semantic checks that might only run on a sample.
- **Monitoring:** logging citation-verification failures separately from other error types preserves an early, cheap warning signal that could catch systemic problems before they're caught by more expensive downstream checks. A rising rate of invalid citations, especially isolated to a specific topic or query pattern, is a strong diagnostic signal worth investigating immediately.
- **Deployment and failure handling:** in a regulated domain, an invalid citation shouldn't silently pass through to the customer. The response should be blocked from customer-facing surfacing and routed to human review, with the specific invalid citation and the full context that was sent logged for both immediate correction and pattern detection.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Structured tool-based citation vs inline natural-language citation:** structured citation, if the surrounding system already has established tool-use infrastructure, reuses that infrastructure and produces a reliably parseable field that's trivial to verify via simple set membership. Inline citation requires parsing free text and is more brittle to variations in how the model happens to phrase its output, but may be preferable for customer-facing surfaces where a clean, readable natural-language answer matters more than perfect programmatic verifiability.
- **Answer-level citation vs sentence-level citation:** answer-level citation (one list of sources for the whole answer) is simpler to implement and sufficient for short, single-fact-focused questions. Sentence-level citation (mapping each individual sentence to its specific source) provides much finer-grained trust signals, but requires a more complex prompt and output schema, is harder for the model to reliably produce correctly, and adds latency and additional failure surface. The added complexity is worth it specifically for longer, multi-fact answers where partial grounding failure (some claims correct, others not) is a real and distinguishable risk — this should be validated with real production data before being adopted broadly, not assumed superior by default.
- **How aggressively to fail on invalid citations:** in a regulated domain, failing safe (blocking the response, routing to human review) is the appropriate default, even though this creates friction and requires human review capacity. A less regulated, lower-stakes use case might tolerate a softer failure mode, like silently omitting the unverifiable claim rather than blocking the entire response.


### 6. Alternatives and When to Use Each

- **No citation at all:** acceptable only for the lowest-stakes, most exploratory use cases — not appropriate for a regulated domain given compliance and dispute-resolution considerations.
- **Inline natural-language citation:** reasonable for customer-facing surfaces where a clean, readable answer matters more than perfect programmatic verifiability, provided a best-effort parsing and verification layer still exists on top of it.
- **Structured tool-based citation:** the right default when the surrounding system already has tool-use infrastructure and needs reliable, programmatic verification for both compliance logging and downstream hallucination detection.
- **Sentence-level attribution:** reserved for scenarios where answer-level citation has been validated as genuinely insufficient — for example, long, multi-fact answers where knowing exactly which part is grounded and which isn't materially changes how a human reviewer or downstream system should act.


### 7. Common Mistakes and Production Failures

- Trusting citations without post-hoc verification against the actual context that was sent — a model can cite a plausible-sounding but incorrect source, and without verification this goes undetected.
- Using citation strings from model output directly in file-system or database operations without validation, creating an injection risk.
- Conflating "the citation is valid" (the source was in context) with "the answer is grounded" (the claim actually matches what the source says) — citation verification is necessary but not sufficient; a deeper semantic grounding check covers the gap this leaves open.
- Designing a citation format the model can't reliably reproduce — for example, requiring detail that chunks don't actually carry as metadata — setting the model up to fail or fabricate that detail.
- Not logging citation-verification failures separately from other error types, losing an early, cheap warning signal that could have caught problems before they reached more expensive downstream checks.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why is citation important for a RAG system, beyond general "trustworthiness"?
  A: Citation makes it possible to programmatically verify whether a model's claim is actually grounded in a specific, checkable source, rather than relying on the claim's surface plausibility. For a regulated domain, this is close to a compliance requirement — customers and reviewers need to trace a stated fact back to the exact source document, not just trust the model's paraphrase.

- Q: What's the difference between citation verification and hallucination detection?
  A: Citation verification is a cheap, syntactic check — does the cited source marker actually appear in the context that was sent to the model? Hallucination detection is a more expensive, semantic check — does the model's claim actually match what the cited source says? A citation can be valid while the claim attributed to it is still wrong or unsupported — citation verification alone doesn't catch this.

**Intermediate**

- Q: Why might a project use structured tool-based citation rather than inline natural-language citation?
  A: If the project already has an established tool-use agent pattern, extending it with a citation-carrying finalize tool reuses existing infrastructure and produces a structured field that's trivial to programmatically verify via simple set membership, versus inline citation which requires parsing free text and is more brittle to variations in how the model phrases its output.

- Q: How would you design verification to distinguish "the model cited a source that was never in context" from "the model's claim doesn't match what the cited source actually says"?
  A: These require two separate checks at two separate cost tiers. The first — citation validity — is a cheap set-membership check: is the cited source marker present in the known set of sources for this specific request? This should run synchronously on every request. The second — semantic grounding — requires a more expensive method like an entailment model, an LLM-as-judge call, or embedding-similarity comparison between the claim and the cited passage; this may run on a sample or asynchronously rather than blocking every response.

**Advanced**

- Q: Design the failure-handling policy for when citation verification detects an invalid citation in a customer-facing response for a regulated domain.
  A: An invalid citation should not silently pass through to the customer. The response should be blocked from customer-facing surfacing and routed to a human-review queue, making the failure explicit and structured rather than silent. The specific invalid citation and the full context that was sent should be logged for review, both to fix the immediate case and to identify whether a systematic prompt or context-construction issue is causing repeated failures. A rising rate of invalid citations for a specific query pattern is also a signal worth feeding back into the retrieval evaluation harness — it may indicate a retrieval gap for that pattern.

- Q: A teammate argues that sentence-level citation is strictly better than answer-level citation and should be the default. How do you respond?
  A: Sentence-level citation provides finer-grained trust signals, but requires a more complex prompt and output schema, is harder for the model to reliably produce correctly, and adds latency and potential failure surface. For short, single-fact-focused questions, answer-level citation likely captures most of the practical value at much lower complexity. Sentence-level citation becomes worth the added cost specifically for longer, multi-fact answers where partial grounding failure is a real, distinguishable risk — this should be validated with actual production data before being adopted as a default, not assumed superior in all cases.

**Scenario-based**

- Q: In production, citation verification starts flagging a rising rate of invalid citations specifically for questions about a newly-launched product. Diagnose.
  A: A spike isolated to a specific new topic strongly suggests either the model is citing a plausible-sounding but non-existent source for content it doesn't have good grounding for, or the actual correct source wasn't retrieved and included in context at all — a retrieval gap, not a generation gap. First check whether documentation for the new product was actually ingested and is being retrieved, using the retrieval evaluation harness extended with queries about the new product. If retrieval is genuinely finding the right chunks but the model still cites incorrectly, the issue is more likely a prompting or model-behavior problem specific to unfamiliar content — worth testing whether being more explicit in the system prompt about strict citation-to-provided-context-only behavior reduces the rate.

**Follow-up questions to expect:**

- "How would you monitor citation quality over time as the knowledge base grows?"
- "What would you do if the model consistently cites the wrong source among several very similar ones?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Citation verification is a special case of output validation:** a general pattern that recurs broadly — structured output schemas already provide a first layer of validation by construction, and citation verification is a second, semantic-adjacent layer built on top of that structural validation.
- **The relationship between citation and formal end-to-end RAG evaluation metrics:** some formal RAG evaluation frameworks measure whether the context that was actually used is relevant — citation data (which sources the model says it used) is a direct input to computing such metrics at scale, connecting this topic's mechanism to more formal evaluation approaches covered later.
- **Citation as an interpretability tool, not just a trust mechanism:** beyond customer-facing trust, citation data aggregated across many requests reveals which knowledge base documents are actually being relied upon in practice versus which are rarely cited despite being retrieved — a useful signal for knowledge base maintenance and content prioritization decisions.

### 10. Quick Revision Summary (for last-minute interview prep)

> Citation attributes each factual claim in a RAG answer to a specific, verifiable source from the retrieved context, closing the gap between "plausible-sounding" and "actually grounded." Structured citation via a tool-based mechanism, when the surrounding system already has tool-use infrastructure, is generally preferable to inline natural-language citation because it produces a reliably parseable field. Citation verification (checking that cited sources actually appeared in the context sent) is a cheap, necessary-but-not-sufficient syntactic check, distinct from the more expensive semantic grounding check that verifies claims actually match their cited sources — a citation can be valid while the underlying claim is still wrong. Invalid citations in a regulated domain should block customer-facing surfacing and route to human review, not silently pass through, and should be logged and monitored as an early warning signal distinct from downstream hallucination metrics.


### Module 1: The Structured Citation Schema and a Simulated Model Response

Define the tool schema and simulate a few model responses (some correct, some with genuine citation bugs) to test verification against.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Structured citation schema + simulated model responses
# ------------------------------------------------------------------

ANSWER_TOOL_SCHEMA = {
    "name": "finalize_answer_with_citations",
    "description": "Call this when you have a complete answer to the customer's question, grounded in the provided context.",
    "input_schema": {
        "type": "object",
        "properties": {
            "answer": {"type": "string", "description": "The final answer to the customer."},
            "sources_used": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Every source marker that was actually used to construct the answer.",
            },
        },
        "required": ["answer", "sources_used"],
    },
}

# the context actually sent for this request -- these are the ONLY
# sources that should ever legitimately be cited
CONTEXT_SOURCES_SENT = {"01_FD_FAQ.pdf", "04_FD_Policy_Reference.pdf", "03_FD_SOPs.pdf"}

# simulated model tool-call outputs for the SAME request -- some
# correct, some with real citation bugs, to test verification against
SIMULATED_MODEL_RESPONSES = [
    {
        "label": "Correct citation",
        "answer": "The penalty for premature withdrawal is 1 percent on the applicable interest rate.",
        "sources_used": ["04_FD_Policy_Reference.pdf"],
    },
    {
        "label": "Citation to a source never sent (hallucinated source)",
        "answer": "The penalty for premature withdrawal is 1 percent, and this was updated in the 2024 circular.",
        "sources_used": ["04_FD_Policy_Reference.pdf", "2024_RBI_Circular.pdf"],  # never in context!
    },
    {
        "label": "No sources cited at all",
        "answer": "The penalty for premature withdrawal is 1 percent on the applicable interest rate.",
        "sources_used": [],
    },
    {
        "label": "Multiple valid sources correctly cited",
        "answer": "Premature withdrawal has a 1 percent penalty; the SOP requires a written closure request.",
        "sources_used": ["04_FD_Policy_Reference.pdf", "03_FD_SOPs.pdf"],
    },
]

print("=" * 70)
print("SIMULATED MODEL RESPONSES TO VERIFY")
print("=" * 70)
print(f"Context sources actually sent: {sorted(CONTEXT_SOURCES_SENT)}\n")
for resp in SIMULATED_MODEL_RESPONSES:
    print(f"  [{resp['label']}]")
    print(f"    answer: {resp['answer'][:60]}...")
    print(f"    sources_used: {resp['sources_used']}")
print("\nModule 1 complete. Run Module 2 next.")


SIMULATED MODEL RESPONSES TO VERIFY
Context sources actually sent: ['01_FD_FAQ.pdf', '03_FD_SOPs.pdf', '04_FD_Policy_Reference.pdf']

  [Correct citation]
    answer: The penalty for premature withdrawal is 1 percent on the app...
    sources_used: ['04_FD_Policy_Reference.pdf']
  [Citation to a source never sent (hallucinated source)]
    answer: The penalty for premature withdrawal is 1 percent, and this ...
    sources_used: ['04_FD_Policy_Reference.pdf', '2024_RBI_Circular.pdf']
  [No sources cited at all]
    answer: The penalty for premature withdrawal is 1 percent on the app...
    sources_used: []
  [Multiple valid sources correctly cited]
    answer: Premature withdrawal has a 1 percent penalty; the SOP requir...
    sources_used: ['04_FD_Policy_Reference.pdf', '03_FD_SOPs.pdf']

Module 1 complete. Run Module 2 next.


### Module 2: Citation Verification

The actual verification function — a cheap, fast, syntactic check that every cited source genuinely appeared in the context sent.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Citation verification (cheap, syntactic check)
# ------------------------------------------------------------------

def verify_citations(sources_used: list, context_sources: set) -> dict:
    """Checks that every cited source marker actually appeared in the
    context sent for this request. This is a set-membership check --
    cheap, fast, and catches a definitive class of model error, but
    does NOT verify the underlying claim is actually correct."""
    cited_set = set(sources_used)
    invalid_citations = cited_set - context_sources
    is_valid = len(invalid_citations) == 0 and len(cited_set) > 0

    return {
        "is_valid": is_valid,
        "invalid_citations": sorted(invalid_citations),
        "has_any_citation": len(cited_set) > 0,
    }


print("=" * 70)
print("CITATION VERIFICATION RESULTS")
print("=" * 70)

for resp in SIMULATED_MODEL_RESPONSES:
    result = verify_citations(resp["sources_used"], CONTEXT_SOURCES_SENT)
    status = "PASS" if result["is_valid"] else "FAIL"
    print(f"\n[{resp['label']}] -> {status}")
    if result["invalid_citations"]:
        print(f"  Invalid citations (source never sent): {result['invalid_citations']}")
    if not result["has_any_citation"]:
        print("  No citations provided at all -- cannot verify grounding.")
    if result["is_valid"]:
        print(f"  All {len(resp['sources_used'])} cited source(s) confirmed present in context.")

print("\nNotice: verification is a CHEAP, SYNTACTIC check. It correctly")
print("catches the hallucinated-source case and the no-citation case,")
print("but it CANNOT tell you whether the CLAIM itself is actually")
print("correct -- that requires the deeper semantic check from Topic 4.")
print("\nModule 2 complete. Run Module 3 next.")


CITATION VERIFICATION RESULTS

[Correct citation] -> PASS
  All 1 cited source(s) confirmed present in context.

[Citation to a source never sent (hallucinated source)] -> FAIL
  Invalid citations (source never sent): ['2024_RBI_Circular.pdf']

[No sources cited at all] -> FAIL
  No citations provided at all -- cannot verify grounding.

[Multiple valid sources correctly cited] -> PASS
  All 2 cited source(s) confirmed present in context.

Notice: verification is a CHEAP, SYNTACTIC check. It correctly
catches the hallucinated-source case and the no-citation case,
but it CANNOT tell you whether the CLAIM itself is actually
correct -- that requires the deeper semantic check from Topic 4.

Module 2 complete. Run Module 3 next.


### Module 3: Failure Handling Policy for a Regulated Domain

Implements the decision logic: what happens to a response depending on its verification outcome, mirroring a fail-safe (not fail-silent) policy.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Failure handling policy
#
# In a regulated domain, an invalid citation should never silently
# pass through to the customer. This module implements that routing
# decision and the associated audit log entry.
# ------------------------------------------------------------------

def route_response(resp: dict, context_sources: set) -> dict:
    """Decides what happens to a generated response based on its
    citation verification outcome -- the actual production policy,
    not just the verification check itself."""
    verification = verify_citations(resp["sources_used"], context_sources)

    if verification["is_valid"]:
        decision = "SURFACE_TO_CUSTOMER"
        reason = "All citations verified against context sent."
    elif not verification["has_any_citation"]:
        decision = "BLOCK_AND_ROUTE_TO_HUMAN_REVIEW"
        reason = "No citations provided -- cannot verify grounding for a regulated-domain answer."
    else:
        decision = "BLOCK_AND_ROUTE_TO_HUMAN_REVIEW"
        reason = f"Invalid citation(s) detected: {verification['invalid_citations']}"

    log_entry = {
        "answer_preview": resp["answer"][:50],
        "sources_used": resp["sources_used"],
        "decision": decision,
        "reason": reason,
    }
    return log_entry


print("=" * 70)
print("PRODUCTION ROUTING DECISIONS (fail-safe, not fail-silent)")
print("=" * 70)

audit_log = []
for resp in SIMULATED_MODEL_RESPONSES:
    log_entry = route_response(resp, CONTEXT_SOURCES_SENT)
    audit_log.append(log_entry)
    print(f"\n[{resp['label']}]")
    print(f"  Decision: {log_entry['decision']}")
    print(f"  Reason:   {log_entry['reason']}")

blocked_count = sum(1 for entry in audit_log if entry["decision"] != "SURFACE_TO_CUSTOMER")
print(f"\n{blocked_count} of {len(audit_log)} simulated responses were blocked from")
print("customer-facing surfacing and routed to human review -- exactly")
print("the fail-safe behavior a regulated domain requires: invalid")
print("citations are made explicit and actionable, never silently ignored.")

print("\nModule 3 complete. All Chapter 8 Topic 2 modules done.")
print()
print("=" * 70)
print("CHAPTER 8 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Citation verification = cheap, syntactic check: did the cited source
  marker actually appear in the context sent? Set-membership, fast,
  runs on every request.

  Citation verification is NECESSARY but NOT SUFFICIENT -- a citation
  can be valid while the underlying claim is still wrong. That deeper
  semantic check is a separate, more expensive concern.

  In a regulated domain: invalid citations should BLOCK customer-facing
  surfacing and route to human review -- fail-safe, never fail-silent.

  Structured tool-based citation (a required sources_used field) is
  more reliable to verify programmatically than inline natural-language
  citation, IF the surrounding system already has tool-use infrastructure.
""")


PRODUCTION ROUTING DECISIONS (fail-safe, not fail-silent)

[Correct citation]
  Decision: SURFACE_TO_CUSTOMER
  Reason:   All citations verified against context sent.

[Citation to a source never sent (hallucinated source)]
  Decision: BLOCK_AND_ROUTE_TO_HUMAN_REVIEW
  Reason:   Invalid citation(s) detected: ['2024_RBI_Circular.pdf']

[No sources cited at all]
  Decision: BLOCK_AND_ROUTE_TO_HUMAN_REVIEW
  Reason:   No citations provided -- cannot verify grounding for a regulated-domain answer.

[Multiple valid sources correctly cited]
  Decision: SURFACE_TO_CUSTOMER
  Reason:   All citations verified against context sent.

2 of 4 simulated responses were blocked from
customer-facing surfacing and routed to human review -- exactly
the fail-safe behavior a regulated domain requires: invalid
citations are made explicit and actionable, never silently ignored.

Module 3 complete. All Chapter 8 Topic 2 modules done.

CHAPTER 8 TOPIC 2 -- KEY POINTS TO REMEMBER

  Citation verification = chea